# Notebook 2 — Linear regression, end to end

We pick up the pipeline from Notebook 1 and take it all the way through: fit, read the
weights, evaluate honestly, regularize, tune, and finally check the answer against the
process that actually generated the data.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

plt.rcParams['figure.figsize'] = (8, 4)

from pathlib import Path

# works whether you run this from the notebooks folder or from a subfolder
DATA = Path('data') if Path('data').exists() else Path('..') / 'data'

season = pd.read_csv(DATA / 'bacterial_spot_season.csv')

NUM = ['wetness_hours', 'mean_temp_c', 'max_temp_c', 'rain_mm',
       'inoculum_log', 'soil_ph', 'dist_to_road_m']
CAT = ['cultivar', 'transplant_source']

X = season[NUM + CAT]
t = season['final_severity']

pre = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), NUM),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), CAT),
])

model = Pipeline([('pre', pre), ('lin', LinearRegression())])
model.fit(X, t)

---
## 1. Read the weights

`coef_` comes back as a bare array. Matching it to column names is a step people skip,
and it is where sign errors hide.

In [ ]:
names = NUM + list(model.named_steps['pre']
                   .named_transformers_['cat'].get_feature_names_out(CAT))
coefs = model.named_steps[#].coef_

weights = pd.Series(coefs, index=names).sort_values()
weights.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#FF4A00' if abs(v) < 1 else '#3677A5' for v in weights]
weights.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='#444444', lw=1)
ax.set_xlabel('Weight (severity %, per standard deviation)')
plt.tight_layout()

Because every numeric column was standardized, these weights are on a common scale: a
one-standard-deviation change in that feature, in severity points.

Two of them (orange) sit almost on zero. Hold that thought.

### Interpreting a weight out loud

The `cultivar_Tygress` weight is about **-20**. That means: relative to the reference
cultivar `FL-8000`, and holding the weather and inoculum columns fixed, Tygress plots
run roughly 20 severity points lower.

The phrase *holding the other columns fixed* is doing real work. It is why a weight is
not the same thing as the relationship you would see in a scatter plot of that one
variable.

---
## 2. Evaluate — in-sample first, then honestly

In [ ]:
pred = model.predict(X)

print(f'RMSE (in-sample) {np.sqrt(mean_squared_error(t, pred)):6.2f}')
print(f'MAE  (in-sample) {mean_absolute_error(t, pred):6.2f}')
print(f'R2   (in-sample) {r2_score(t, pred):6.2f}')

In [ ]:
loyo = GroupKFold(10)
rmse_loyo = -cross_val_score(model, X, t, cv=loyo, groups=#,
                             scoring='neg_root_mean_squared_error').mean()
baseline = np.sqrt(((t - t.mean()) ** 2).mean())

print(f'RMSE leaving out a season {rmse_loyo:6.2f}')
print(f'RMSE predicting the mean  {baseline:6.2f}')

The honest headline is the middle number, not the first one. Against a baseline of
about 24 severity points, the model earns its keep — but it is not a triumph, and
that is a realistic outcome for field epidemiology.

---
## 3. Look at the residuals

Four different failures can produce a similar R². Only the plot tells them apart.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(t, pred, s=18, alpha=0.5, color='#3677A5')
axes[0].plot([0, 100], [0, 100], '--', color='#FF4A00', lw=2)
axes[0].set_xlabel('Observed severity (%)')
axes[0].set_ylabel('Predicted severity (%)')

resid = #
axes[1].scatter(pred, resid, s=18, alpha=0.5, color='#3677A5')
axes[1].axhline(0, color='#FF4A00', lw=2)
axes[1].set_xlabel('Predicted severity (%)')
axes[1].set_ylabel('Residual')

plt.tight_layout()

### What do you see?

Look at the left panel at the extremes. Observed severities crowd against 0 and 100;
the predictions do not. The model happily predicts values a bounded percentage cannot
take.

That is the ceiling effect from the slides, visible in our own fit. It is the honest
argument for a bounded model — a GLM or a beta regression — rather than plain least
squares.

---
## 4. Regularization

Ridge shrinks every weight toward zero. Lasso can set weights exactly to zero.

Watch what happens to those two features that were sitting near zero.

In [ ]:
ridge = Pipeline([('pre', pre), ('m', Ridge(alpha=30))]).fit(X, t)
lasso = Pipeline([('pre', pre), ('m', Lasso(alpha=#))]).fit(X, t)

comparison = pd.DataFrame({
    'no penalty': coefs,
    'ridge': ridge.named_steps['m'].coef_,
    'lasso': #,
}, index=names)

comparison.round(2)

In [ ]:
zeroed = comparison.index[comparison.lasso.abs() < 1e-8].tolist()
print('Lasso set these to exactly zero:')
for z in zeroed:
    print('  -', z)

Remember those names. We will find out shortly whether lasso was right.

Note that it removed **five** things, not two. Two of them are cultivar effects that
are small but genuinely non-zero. Lasso does not distinguish "zero" from "small" — it
shrinks until something disappears, and small real effects go over the edge with the
noise. That is the price of the feature selection.

---
## 5. Choose the penalty by cross-validation, not by eye

`alpha` is a hyperparameter: the model cannot learn it from the same data it fits. We
search over a grid, scoring with the same leave-one-year-out scheme we trust.

In [ ]:
grid = GridSearchCV(
    Pipeline([('pre', pre), ('m', Ridge())]),
    param_grid={'m__alpha': #},
    cv=GroupKFold(10),
    scoring='neg_root_mean_squared_error',
)
grid.fit(X, t, groups=#)

print('best alpha:', grid.best_params_)
print(f'best RMSE : {-grid.best_score_:.2f}')

In [ ]:
res = pd.DataFrame(grid.cv_results_)[['param_m__alpha', 'mean_test_score']]
res['RMSE'] = -res.mean_test_score
res[['param_m__alpha', 'RMSE']].round(3)

Note how flat that curve is. With only seven numeric features and no severe
collinearity beyond one pair, there is not much for the penalty to do. Regularization
earns its keep when you have forty weather summaries, not seven.

---
## 6. The reveal

This dataset was generated from a process we can read. Let us see what the model
recovered — and what it invented.

In [ ]:
truth = json.load(open(DATA / 'truth.json'))
tc = truth['season_model']['coefficients']

print('TRUE generating coefficients (per raw unit):')
for k in ['wetness_hours', 'mean_temp_c', 'max_temp_c', 'rain_mm', 'inoculum_log',
          'soil_ph', 'dist_to_road_m']:
    print(f'  {k:16s} {tc[k]:+.4f}')
print()
print('TRUE cultivar effects:')
for k, v in truth['season_model']['cultivar_effect'].items():
    print(f'  {k:10s} {v:+.1f}')

### Three things to check

1. **`soil_ph` and `dist_to_road_m` are exactly zero in the truth.** They are noise
   columns we planted. Did lasso remove them?
2. **`max_temp_c` is exactly zero in the truth too** — but it correlates about 0.93
   with `mean_temp_c`. Did the model give it a weight anyway?
3. **Cultivar ordering.** Tygress is the most resistant, then Sanibel, then Sebring;
   HM-1823 is *more* susceptible than the reference.

In [ ]:
print('correlation between mean_temp_c and max_temp_c: '
      f'{season.mean_temp_c.corr(season.max_temp_c):.3f}')
print()
print('fitted weight for max_temp_c (true effect = 0): '
      f'{weights[#]:+.2f}')
print('fitted weight for soil_ph    (true effect = 0): '
      f'{weights[#]:+.2f}')

`max_temp_c` has no causal role whatsoever, and the model gave it a clearly non-zero
weight. Nothing went wrong: two nearly identical columns split the credit between them,
and least squares has no way to know which one is the real one.

This is the single most important caveat about coefficient tables, and it does not go
away with more data. **A weight is not an effect.**

---
## 7. Final evaluation, once

We have used cross-validation to choose things. Now score the chosen model on a season
it has never been fitted on, and report that number.

In [ ]:
train = season[season.year < 2025]
test = season[season.year == 2025]

final = Pipeline([('pre', pre), ('m', Ridge(alpha=grid.best_params_['m__alpha']))])
final.fit(train[NUM + CAT], train.final_severity)

y_test = final.predict(test[NUM + CAT])
rmse_test = np.sqrt(mean_squared_error(test.final_severity, y_test))

print(f'held-out season 2025: n = {len(test)}, RMSE = {rmse_test:.2f}')
print(f'baseline on that season:        RMSE = '
      f'{np.sqrt(((test.final_severity - train.final_severity.mean())**2).mean()):.2f}')

One season is only 48 field-seasons, so this number carries real uncertainty. Report it
with the cross-validated estimate, not instead of it.

---
## 8. What to take from this notebook

* Match `coef_` to column names deliberately; do not trust the ordering by eye.
* Standardize if you want to compare weights across features.
* The in-sample R² and the leave-one-season-out RMSE tell different stories. Report
  the second.
* Lasso removed the two noise columns — and left the collinear decoy in place.
* Correlated predictors split credit. A weight is a prediction ingredient, not an
  effect estimate.

**Next**: Notebook 3 changes the target from a number to a decision.